# Dá pra morar em Lucas? Preço dos imóveis x renda — a conta da acessibilidade

**Autora:** Thamires Martins da Silva
**Fontes:** Dataset de preços da Camada 3 (236 anúncios) · Salário mínimo 2026 · Rendimento médio de MT (PNAD Contínua/IBGE, 2024)
**Ferramentas:** Python (pandas, matplotlib)

---

## Contexto

As três primeiras camadas desta análise sobre Lucas do Rio Verde mostraram uma cidade sob pressão: população crescendo rápido (Camada 1), oferta de moradia aquém (Camada 2) e preços em patamar de capital (Camada 3). Fica uma pergunta que nenhuma delas respondeu, e que é a que mais importa para quem vive na cidade: **com a renda local, quem consegue de fato comprar um imóvel?**

Esta camada cruza o preço dos imóveis com a renda para medir a *acessibilidade* — quanto custa um imóvel em anos de renda, e que renda familiar é preciso ter para financiá-lo.

> **Pergunta de negócio:** Para diferentes faixas de renda familiar de referência, que parcela dos imóveis anunciados poderia caber no orçamento de uma simulação de financiamento?

## Nota de método sobre a renda

O IBGE não publica uma renda familiar municipal fechada para Lucas, então a análise não crava um único número de renda local. Em vez disso, ela calcula a acessibilidade **em função da renda** — para uma faixa de rendas familiares, do salário mínimo para cima — e usa como âncoras oficiais o salário mínimo de 2026 (R\$ 1.621) e o rendimento médio mensal de Mato Grosso (R\$ 3.402 por pessoa, PNAD Contínua/IBGE, 2024). Essa abordagem é, inclusive, mais útil na prática: é assim que um corretor pensa o cliente — por faixa de renda.

**Qualidade da base:** a amostra da Camada 3 contém 233 registros após a remoção de três linhas integralmente duplicadas. Combinações iguais entre fontes diferentes foram mantidas para revisão, sem exclusão automática. As estimativas de renda e financiamento são cenários com premissas explícitas, não ofertas de crédito nem retratos exatos da renda familiar municipal.

## 1. Configuração e carga

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

AZUL_ESC='#1a3a5c'; AZUL='#2c7fb8'; TEAL='#7fcdbb'; LARANJA='#e8703a'; CINZA='#8a9299'

def reais(v): return ('R$ '+format(int(round(v)),',d')).replace(',','.')

SALARIO_MINIMO = 1621          # 2026
REND_MEDIO_MT  = 3402          # PNAD Continua/IBGE 2024, por pessoa

df = pd.read_csv(Path.cwd()/'precos_imoveis_lucas.csv', sep=';')
duplicatas_exatas = int(df.duplicated().sum())
if duplicatas_exatas:
    df = df.drop_duplicates().copy()

# Combinações iguais entre fontes diferentes são candidatas a revisão,
# mas não são removidas automaticamente sem identificação do imóvel.
chave_candidata = ['bairro', 'tipo', 'area_m2', 'preco_total', 'preco_m2']
candidatos_duplicacao = df[df.duplicated(subset=chave_candidata, keep=False)]
print(f'Duplicatas exatas removidas ao carregar: {duplicatas_exatas}')
print(f'Registros com possível sobreposição entre fontes: {len(candidatos_duplicacao)}')

tip = df[df['atipico']=='nao'].copy()
preco_tipico = tip['preco_total'].median()
print(f'Anuncios tipicos: {len(tip)}')
print(f'Imovel tipico (mediana):    {reais(preco_tipico)}')
print(f'  casa:  {reais(tip[tip.tipo=="casa"]["preco_total"].median())}')
print(f'  apto:  {reais(tip[tip.tipo=="apartamento"]["preco_total"].median())}')
print(f'Salario minimo 2026:        {reais(SALARIO_MINIMO)}')
print(f'Rendimento medio MT/pessoa: {reais(REND_MEDIO_MT)}')

## 2. Quantos anos de renda custa um imóvel

In [ ]:
def anos_de_renda(preco, renda_mensal):
    return preco/(renda_mensal*12)

print('=== ANOS DE RENDA FAMILIAR INTEGRAL PARA O IMOVEL TIPICO ===')
print('   (renda inteira, sem descontar nenhum outro gasto)\n')
faixas_sm=[2,3,4,6,8,10]
for sm in faixas_sm:
    renda=sm*SALARIO_MINIMO
    print(f'  {sm:2} sal.min.  ({reais(renda)}/mes)  ->  {anos_de_renda(preco_tipico,renda):4.1f} anos')
print()
print(f'Referencia: ao rendimento medio individual de MT ({reais(REND_MEDIO_MT)}),')
print(f'o imovel tipico equivale a {anos_de_renda(preco_tipico,REND_MEDIO_MT):.0f} anos de renda.')

In [ ]:
rendas=np.array([2,3,4,5,6,8,10])*SALARIO_MINIMO
anos=[anos_de_renda(preco_tipico,r) for r in rendas]
fig,ax=plt.subplots(figsize=(11,5.5))
cores=[LARANJA if a>5 else TEAL for a in anos]
ax.bar([f'{int(r/SALARIO_MINIMO)} SM' for r in rendas],anos,color=cores,zorder=3)
for i,a in enumerate(anos):
    ax.annotate(f'{a:.1f} anos',(i,a),textcoords='offset points',xytext=(0,6),ha='center',
                fontsize=10,fontweight='bold',color=AZUL_ESC)
ax.axhline(5,color=CINZA,linestyle='--',linewidth=1.5,zorder=2)
ax.annotate('Linha de 5 anos de renda:\nacima disso, mercado severamente inacessivel\n(referencia internacional Demographia)',
            (len(rendas)-1,5),textcoords='offset points',xytext=(-6,12),ha='right',fontsize=8.5,color=CINZA)
ax.set_title('Quantos anos de renda familiar integral custa o imovel tipico de Lucas',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Renda familiar (em salarios minimos)'); ax.set_ylabel('Anos de renda integral')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
plt.tight_layout(); plt.show()

### Leitura

A referência internacional (índice Demographia) considera um mercado "severamente inacessível" quando o imóvel custa mais de 5 anos de renda familiar. Em Lucas, **só famílias a partir de 6 salários mínimos cruzam essa linha** — e ainda assim gastando a renda inteira por mais de 4 anos. Para a maioria, que ganha menos, o imóvel típico está a 7, 9, 14 anos de renda. O preço, que a Camada 3 mostrou ser de capital, encontra aqui o seu efeito social: ele está acima do alcance da renda local típica.

## 3. A conta real: financiar o imóvel típico

In [ ]:
def parcela_price(valor_financiado, taxa_aa=0.11, meses=360):
    i=(1+taxa_aa)**(1/12)-1
    return valor_financiado*i/(1-(1+i)**-meses)

ENTRADA=0.20
financiado=preco_tipico*(1-ENTRADA)
print(f'Imovel tipico: {reais(preco_tipico)}')
print(f'Entrada (20%): {reais(preco_tipico*ENTRADA)}   |   Financiado: {reais(financiado)}')
print(f'Prazo: 360 meses (30 anos)   |   Sistema: Price\n')
print('=== PARCELA E RENDA NECESSARIA (por taxa de juros) ===')
for taxa in [0.10,0.11,0.12]:
    p=parcela_price(financiado,taxa)
    renda_nec=p/0.30   # parcela <= 30% da renda
    print(f'  taxa {int(taxa*100)}% a.a.  ->  parcela {reais(p)}/mes  ->  renda familiar necessaria {reais(renda_nec)}  ({renda_nec/SALARIO_MINIMO:.1f} sal.min.)')

In [ ]:
taxas=[0.10,0.11,0.12]
parcelas=[parcela_price(financiado,t) for t in taxas]
rendas_nec=[p/0.30 for p in parcelas]
fig,ax=plt.subplots(figsize=(11,5.5))
x=np.arange(len(taxas))
ax.bar(x,rendas_nec,color=AZUL,width=.55,zorder=3)
for i,(rn,p) in enumerate(zip(rendas_nec,parcelas)):
    ax.annotate(f'{reais(rn)}\n(parcela {reais(p)})',(i,rn),textcoords='offset points',xytext=(0,6),
                ha='center',fontsize=10,fontweight='bold',color=AZUL_ESC)
ax.set_xticks(x); ax.set_xticklabels([f'{int(t*100)}% a.a.' for t in taxas])
ax.set_title('Renda familiar necessaria para financiar o imovel tipico (R$ 540 mil)',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Taxa de juros do financiamento'); ax.set_ylabel('Renda familiar necessaria (R$/mes)')
ax.set_ylim(0,max(rendas_nec)*1.25)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,p:f'{v/1000:.0f}k'))
plt.tight_layout(); plt.show()

### Leitura

Esta é a conta que o cliente faz no balcão. Para financiar o imóvel típico, a família precisa de uma renda em torno de **R\$ 13 mil por mês — cerca de 8 salários mínimos** — só para a parcela caber nos 30% recomendados, e isso depois de ter juntado R\$ 108 mil de entrada. Mudar a taxa de juros de 10% para 12% ao ano move a renda necessária em mais de R\$ 1,9 mil. É um patamar que exclui a maior parte dos assalariados da cidade e ajuda a explicar, do lado da demanda, por que tanto imóvel fica anunciado: o preço pede uma renda que poucos têm.

## 4. E o que cabe em cada renda?

In [ ]:
def maior_imovel(renda_mensal, ENTRADA=0.20, taxa=0.11):
    # maior valor de imovel cuja parcela cabe em 30% da renda, dada a entrada
    parcela_max=0.30*renda_mensal
    i=(1+taxa)**(1/12)-1
    financiado_max=parcela_max*(1-(1+i)**-360)/i
    return financiado_max/(1-ENTRADA)

print('=== TETO DE IMOVEL FINANCIAVEL POR RENDA (entrada 20%, 11% a.a., 30 anos) ===\n')
for sm in [2,3,4,6,8,10]:
    renda=sm*SALARIO_MINIMO
    teto=maior_imovel(renda)
    pct=(tip['preco_total']<=teto).mean()*100
    print(f'  {sm:2} SM ({reais(renda)}/mes)  ->  ate {reais(teto):>14}   | alcanca {pct:4.0f}% dos imoveis da amostra')

### Leitura

Virando a conta do outro lado: com 4 salários mínimos de renda familiar, o teto de imóvel financiável fica bem abaixo do imóvel típico, alcançando só uma fatia pequena da amostra — e são justamente as unidades mais compactas e os bairros mais populares que a Camada 3 mapeou. Só a partir de 8 salários mínimos a maior parte do mercado se abre. É um retrato de um mercado segmentado por renda: para a base da pirâmide salarial, sobra a ponta mais barata do pátio; o miolo e o topo são para poucos.

## 5. Limitações

1. **Preço de anúncio, não de venda.** O valor usado é o pedido (Camada 3), geralmente acima do fechamento — a acessibilidade real é um pouco melhor que a calculada, mas na mesma ordem de grandeza.
2. **Sem renda familiar municipal fechada.** O IBGE não publica esse número para Lucas; por isso a análise é feita por faixa de renda, ancorada no salário mínimo e no rendimento médio de MT. Não é um retrato da renda exata da cidade, e sim da relação preço/renda.
3. **Parâmetros de financiamento assumidos** (entrada 20%, 30 anos, 10-12% a.a., sistema Price, parcela em 30% da renda). São premissas padrão de mercado; condições reais variam por banco, programa (ex.: Minha Casa Minha Vida) e perfil.
4. **Renda familiar, não individual.** Os cenários supõem a renda somada da família; quantos ganham por domicílio varia.

## 6. Síntese executiva

Lucas do Rio Verde tem preços de imóvel em patamar de capital — e a renda local não acompanha. O imóvel típico, de **R\$ 540 mil**, exige para ser financiado uma renda familiar de cerca de **R\$ 13 mil mensais (perto de 8 salários mínimos)**, além de uma entrada de R\$ 108 mil. Medido em anos de renda, ele custa de **3,5 anos** (para uma família de 8 SM) a quase **14 anos** (para uma de 2 SM) de renda integral — muito acima da linha de 5 anos que marca, internacionalmente, um mercado severamente inacessível.

O efeito é um mercado nitidamente segmentado: para quem ganha na média dos assalariados da cidade, o que cabe no bolso é a ponta mais barata e mais afastada do mercado; o miolo e o alto padrão ficam reservados a uma minoria de alta renda — em boa parte ligada ao agro, o motor que puxou a população (Camada 1) e os preços (Camada 3).

Fecha-se o ciclo das quatro camadas com uma leitura coerente: **a cidade cresceu, construiu menos do que precisava, os preços subiram a nível de capital — e, no fim da conta, morar na própria cidade ficou caro demais para a renda de quem a faz girar.**

---

*Quarta e última camada da análise sobre habitação em Lucas do Rio Verde. Baseada no dataset de preços da Camada 3 (236 anúncios) e em dados oficiais de renda (salário mínimo 2026; PNAD Contínua/IBGE 2024 para Mato Grosso).*